![Redis](https://redis.io/wp-content/uploads/2024/04/Logotype.svg?auto=webp&quality=85,75&width=120)

# Persistent Context Across Conversations with Redis Agent Memory
Memory is essential to agentic systems because it enables continuity, learning, and context-aware action across sessions. With [Redis Agent Memory](https://redis.io/agent-memory/), agents gain intelligent short-term memory within the session and persistent context across conversations, allowing their intelligence and relevance to compound over time.

Redis Agent Memory combines short-term memory (which gives agents the working context they need to stay coherent and effective within an active session) with long-term memory (which preserves important knowledge, experiences and preferences for future recall). Together, they enable agents to retrieve relevant context when needed and deliver increasingly personalised and context-aware interactions over time, maintaining continuity and ensuring new agentic sessions do not start cold. 

## Before we get started
To understand more about Redis Agent Memory, refer to the following: 
- [Documentation](https://redis.io/docs/latest/operate/iris/agent-memory/) - Documentation for Redis Agent Memory.
- [Managing long and short term memory](https://redis.io/blog/build-smarter-ai-agents-manage-short-term-and-long-term-memory-with-redis/) - Blog post about using Redis Agent Memory to manage long and short term memory in AI Agents.

## Let's Begin!

<a href="https://colab.research.google.com/github/redis-developer/redis-ai-resources/blob/main/python-recipes/agent-memory/01_intro_to_redis_agent_memory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


> **⚠️ This notebook requires Redis Cloud.**
>
> Unlike most recipes in this repo, Redis Agent Memory is a **managed Redis Cloud service** (also available self-managed on Kubernetes in private preview). It does **not** run against a local `redis-stack` container, so this notebook needs an API key from Redis Agent Memory.
>
> You can set up a free account and get an API key by:
> 1. Signing up at [redis.io/try-free](https://redis.io/try-free/?rcplan=iris) (choose the Iris plan).
> 2. Creating an API key in the [Agent Memory console](https://cloud.redis.io/#/agent-memory).

## Setup

First, let's download the required packages and set our credentials:

In [1]:
%pip install "redis-agent-memory>=0.4.1" openai

Note: you may need to restart the kernel to use updated packages.


### Credentials

> Before continuing, create an eligible **Redis Cloud database** and a **Redis Agent Memory service** by following the [setup guide](https://redis.io/docs/latest/operate/iris/agent-memory/create-service).

This notebook needs three values from your Redis Agent Memory service:

- **`AGENT_MEMORY_ENDPOINT`** — The base address of the Redis Agent Memory service, including `https://`.
- **`AGENT_MEMORY_STORE_ID`** — The ID of the memory store used by the service.
- **`AGENT_MEMORY_API_KEY`** — The API key used to authenticate requests. Keep it secret.

Copy the endpoint and store ID from the service's **Configuration** tab in Redis Cloud. The API key is displayed when you create or rotate it, so save it securely.

Additionally, an **`OPENAI_API_KEY`** is required for agent calls.

In [1]:
#NBVAL_SKIP
import os, getpass

AGENT_MEMORY_API_KEY = os.getenv("AGENT_MEMORY_API_KEY") or getpass.getpass("Redis Agent Memory API key: ")
AGENT_MEMORY_STORE_ID = os.getenv("AGENT_MEMORY_STORE_ID") or getpass.getpass("Redis Agent Memory Store ID: ")
AGENT_MEMORY_ENDPOINT = os.getenv("AGENT_MEMORY_ENDPOINT") or getpass.getpass("Redis Agent Memory Endpoint URL: ")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY") or getpass.getpass("OpenAI API key: ")

## Understanding Redis Agent Memory

Redis Agent Memory keeps two kinds of context:

- **Session memory** stores the ordered events of one conversation. Retrieve a session to reconstruct what was said.
- **Long-term memory** stores information that should remain available after that conversation ends, such as goals and preferences.

The names below are the REST API fields. The Python SDK uses the same concepts in snake case, such as `session_id` and `actor_id`. Serialized responses still use camel case. See [Identify users and conversations](https://redis.io/docs/latest/develop/ai/context-engine/agent-memory/developer-guide/#identify-users-and-conversations) in the developer guide.

### Identifiers

These identifiers connect a person, a conversation, and a stored memory:

| Identifier | What it identifies |
| --- | --- |
| `sessionId` | One conversation or interaction. Use a stable ID for the whole conversation. |
| `actorId` | The participant that produced a session event, such as a user or an assistant. |
| `ownerId` | The user or entity a long-term memory belongs to. Filter on this ID so one person's memories are not returned for someone else. |
| Memory ID | One long-term memory within the store. |

The same user identifier can be used for both `actorId` and `ownerId`. They still describe different relationships: `actorId` records who produced an event, while `ownerId` records who a durable memory belongs to.

### Session event fields

A session is an ordered sequence of events. Each event can include:

| Field | Purpose |
| --- | --- |
| `eventId` | Server-generated identifier for the event. |
| `sessionId` | Session that contains the event. |
| `actorId` | Actor that produced the event. |
| `role` | Role of the message, such as user, assistant, or system. |
| `content` | Message content, including its text. |
| `createdAt` | Time the event occurred in the application. |
| `systemTimestamp` | Time Redis Agent Memory stored the event. |
| `metadata` | Optional application-specific information. |

Before an agent turn, retrieve the session by `sessionId` and pass the relevant events back to the agent.

### Long-term memory fields

A long-term memory is a durable record that can be classified, scoped, and searched:

| Field | Purpose |
| --- | --- |
| `id` | Unique identifier for the memory. |
| `text` | Memory content used for retrieval. |
| `memoryType` | Built-in or custom memory type. |
| `sessionId` | Session associated with the memory. |
| `ownerId` | User or entity associated with the memory. |
| `namespace` | Logical grouping for the memory. |
| `topics` | Topic tags used to categorize the memory. |
| `createdAt` | Time the memory was created. |
| `updatedAt` | Time the memory was last updated. |

Redis Agent Memory can create these records automatically by extracting durable information from session events. An application can also create them directly. Search results should be scoped with `ownerId` when memories must stay isolated to one user.

### Memory types

`memoryType` classifies a long-term memory. The built-in types are:

| Type | What it holds |
| --- | --- |
| `semantic` | Durable facts and preferences that remain useful across sessions. |
| `episodic` | A snapshot of what happened in one session. |
| `message` | A built-in type, alongside `semantic` and `episodic`, used to classify and filter records. |
| `session_summary_view` | A built-in type, alongside `semantic` and `episodic`, used to classify and filter records. |

A custom type captures structured, domain-specific information. Define it in the service configuration, with its own name, fields, and extraction prompt. That name is stored in `memoryType` and cannot match a built-in type.

The automatic summary of a long session is returned with the session itself. It is derived session state, separate from these long-term memory types. See [Memory types](https://redis.io/docs/latest/develop/ai/context-engine/agent-memory/overview/#memory-types) in the overview.


### Memory configuration

These settings are on the Agent Memory service in Redis Cloud Console, under [Memory configuration](https://redis.io/docs/latest/operate/iris/agent-memory/create-service/#memory-configuration).

| Setting | What it controls | Default |
| --- | --- | --- |
| Extraction cadence | How often session events are extracted into long-term memory. The range is 60–600 seconds. | 5 minutes |
| Session-memory TTL | How long a session remains available. | 1 day |
| Long-term-memory TTL | How long a long-term memory remains available. | 365 days |

Custom memory types are defined in the same configuration. A new service has none. Each type has a name, fields, and an extraction prompt, and its extraction strategy is enabled by default. Up to three custom types can be enabled.

### Summarization

Automatic summarization is on by default when you create a service in the Redis Cloud Console. Two settings control it:

- **Summarize after (messages)** defaults to 20. That is how many messages a session can hold before older messages are summarized. Once a session passes that count, the oldest messages beyond the keep-most-recent count are compressed into a summary.
- **Keep most recent (messages)** defaults to 10. That is how many of the newest messages stay in full and are never summarized.

With those defaults, a session of 20 messages keeps the latest 10 in full and summarizes the oldest 10. The summary is returned with the session.

### Sensitive-data exclusions

Sensitive-data exclusions keep specified information out of long-term memory during automatic extraction. A new service leaves all three mechanisms off. Session events stay as sent.

- **Semantic exclusions** take a plain-language prompt of up to 2,000 characters, for concepts a pattern cannot catch, such as passwords or recovery codes. Useful context can remain as a generalized memory. The prompt is advisory: session content still reaches the extraction model, and a memory created directly through the API is stored as written.
- **Built-in detectors** match credit card numbers, email addresses, IP addresses, phone numbers, and US Social Security numbers. None are selected by default.
- **Custom detectors** are RE2 patterns you define, such as an internal employee id. A new service has none. A store can have up to 32, each pattern up to 512 characters.

A detector's action defaults to `redact`, which masks the matched text. `drop` discards the candidate memory. When one matching detector is `drop`, that memory is dropped.

## Step 1: Connecting to Redis Agent Memory

### Via SDK

The [redis-agent-memory](https://pypi.org/project/redis-agent-memory/) Python SDK contains the API for storing, retrieving, and searching session memory and long-term memory from Redis Agent Memory. 

In [2]:
#NBVAL_SKIP
from redis_agent_memory import AgentMemory

with AgentMemory(
    AGENT_MEMORY_ENDPOINT,
    store_id=AGENT_MEMORY_STORE_ID,
    api_key=AGENT_MEMORY_API_KEY,
) as memory_client:
    print(memory_client.health())

status=<HealthStatus.HEALTHY: 'healthy'>


### Via HTTPS REST API

Alternatively, Redis Agent Memory can also be accessed via REST API.

In [ ]:
#NBVAL_SKIP

# Using subprocess.run to run a curl request 
import subprocess

health_check = subprocess.run(["curl", 
                                "--fail-with-body", 
                                "--silent", 
                                "--show-error",
                                "--header", 
                                f"Authorization: Bearer {AGENT_MEMORY_API_KEY}", 
                                f"{AGENT_MEMORY_ENDPOINT}/health"],
                                check=True)


{"status":"healthy"}


## Step 2: Recording a conversation in session memory

This notebook follows a fictional scenario of a banking relationship-manager agent and a client. Session memory and long-term memory are used here to generate tailored and targeted responses for clients, according to their preferences and customer profile (e.g. variable income, requires home deposit, etc).

Redis Agent Memory stores the context from prior conversation sessions, and retrieves them in this notebook for:
- **Session memory**: Record previous sessions as an ordered sequence of the conversation between the client and the agent.
- **Long-term memory**: Extract durable details such as the savings goal, timeline, and income constraint, so the next meeting already has that context.
- **Scoped recall**: Retrieve those memories for this client only, so one client's context stays separate from another's.

In [ ]:
#NBVAL_SKIP
from datetime import datetime, timezone
from uuid import uuid4

from redis_agent_memory import models

# Generate a unique run ID to make this notebook safe to rerun.
RUN_ID = uuid4().hex[:8]

# Customer ID is used as the unique identifier for the customer.  
# It will be used as ownerId when searching long-term memory.
CUSTOMER_ID = f"customer-{RUN_ID}"
SESSION_ID = f"banking-session-{RUN_ID}"
ASSISTANT_ID = "relationship-manager-agent"

print(f"Customer: {CUSTOMER_ID}")
print(f"Session: {SESSION_ID}")

Customer: customer-5a9db27a
Session: banking-session-5a9db27a


### Adding session memory

#### Via SDK

The Python SDK can be used to send conversation memory in the current session to Redis Agent Memory. After each turn, call `add_session_event` with the same `session_id` for the whole conversation, the `actor_id` of whoever spoke, and a timezone-aware `created_at`.

In [ ]:
#NBVAL_SKIP
from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)

system_prompt = """You are a relationship-manager agent for a retail bank. You speak with clients about their goals and help them prepare for the next review.

Follow these rules:
- Respond in two or three sentences, in plain language.
- Use only what the client has told you in this conversation.
- Do not invent account balances, transactions, product eligibility, or approval decisions. Those remain in the bank's systems of record.
- Do not give personalised financial, tax, or legal advice. If the client asks for a decision, say a banker needs to confirm it.
"""

history = [{"role": "system", "content": system_prompt}]
conversation = []

client_messages = [
    "I'm trying to pull together a deposit for a place of my own. I'd like to be ready in about a year and a half, but freelance work makes that messy. Some months are great, and some are really quiet.",
]

# Simulate a conversation between the client and the agent
for text in client_messages:
    history.append({"role": "user", "content": text})
    reply = client.chat.completions.create(
        model="gpt-6-luna",
        messages=history,
    ).choices[0].message.content
    history.append({"role": "assistant", "content": reply})

    conversation.append((models.MessageRole.USER, CUSTOMER_ID, text))
    conversation.append((models.MessageRole.ASSISTANT, ASSISTANT_ID, reply))
    print(f"Client: {text}")
    print(f"Agent: {reply}\n")

with AgentMemory(
    AGENT_MEMORY_ENDPOINT,
    store_id=AGENT_MEMORY_STORE_ID,
    api_key=AGENT_MEMORY_API_KEY,
) as memory_client:
    for role, actor_id, text in conversation:
        event = memory_client.add_session_event(
            session_id=SESSION_ID,
            actor_id=actor_id,
            role=role,
            content=[models.Text(text=text)],
            created_at=datetime.now(timezone.utc),
        )
        print(f"Stored {role.value} event: {event.event.event_id}")

Client: I'm trying to pull together a deposit for a place of my own. I'd like to be ready in about a year and a half, but freelance work makes that messy. Some months are great, and some are really quiet.
Agent: That makes sense—freelance income can make an 18-month goal harder to plan around. For your next review, we can talk through the deposit amount you’re aiming for and how your income varies; do you have a rough target in mind?

Stored USER event: 21c5b8ccefbd439299b24e2a9010527c
Stored ASSISTANT event: e2f90d69878447fe94e97559828a4c63


#### Via HTTP POST

Session memory can also be stored in Redis Agent Memory via HTTP POST requests, if the code recording the turn is not a Python process using `redis-agent-memory`.

In [6]:
#NBVAL_SKIP
import json

created_at = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

payload = {
    "sessionId": SESSION_ID,
    "actorId": CUSTOMER_ID,
    "role": "USER",
    "content": [
        {
            "text": "I found a flat I like, but I want to keep the deposit goal flexible until my next busy freelance month."
        }
    ],
    "createdAt": created_at,
}

add_event = subprocess.run(
    [
        "curl",
        "--fail-with-body",
        "--silent",
        "--show-error",
        "--request", "POST",
        "--header", f"Authorization: Bearer {AGENT_MEMORY_API_KEY}",
        "--header", "Content-Type: application/json",
        "--data", json.dumps(payload),
        f"{AGENT_MEMORY_ENDPOINT.rstrip('/')}/v1/stores/{AGENT_MEMORY_STORE_ID}/session-memory/events",
    ],
    check=True,
)

print(f"Stored a user event in session: {SESSION_ID}")

{"event":{"actorId":"customer-5a9db27a","content":[{"text":"I found a flat I like, but I want to keep the deposit goal flexible until my next busy freelance month."}],"createdAt":"2026-10-06T06:07:56Z","eventId":"d4eedee5c85f424ea8a3a1b8fba68ceb","role":"USER","sessionId":"banking-session-5a9db27a","systemTimestamp":"2026-10-06T06:07:57.394Z"}}
Stored a user event in session: banking-session-5a9db27a


### Retrieving session memory

Similarly, the Python SDK or HTTP requests can be used to retrieve memory recorded in the current session.

#### Via SDK

In [7]:
#NBVAL_SKIP
with AgentMemory(
    AGENT_MEMORY_ENDPOINT,
    store_id=AGENT_MEMORY_STORE_ID,
    api_key=AGENT_MEMORY_API_KEY,
) as memory_client:
    session = memory_client.get_session_memory(session_id=SESSION_ID)

print(f"Session: {session.session_id}")
print(f"Owner: {session.owner_id}")
print(f"{len(session.events)} events\n")

# Parse the session memory
for index, event in enumerate(session.events, start=1):
    text = "\n".join(part.text for part in event.content)
    print(f"{index}. {event.role.value} ({event.actor_id})")
    print(text)
    print()

Session: banking-session-5a9db27a
Owner: customer-5a9db27a
3 events

1. USER (customer-5a9db27a)
I'm trying to pull together a deposit for a place of my own. I'd like to be ready in about a year and a half, but freelance work makes that messy. Some months are great, and some are really quiet.

2. ASSISTANT (relationship-manager-agent)
That makes sense—freelance income can make an 18-month goal harder to plan around. For your next review, we can talk through the deposit amount you’re aiming for and how your income varies; do you have a rough target in mind?

3. USER (customer-5a9db27a)
I found a flat I like, but I want to keep the deposit goal flexible until my next busy freelance month.



#### Via HTTP GET

In [8]:
#NBVAL_SKIP
retrieve = subprocess.run(
    [
        "curl",
        "--fail-with-body",
        "--silent",
        "--show-error",
        "--header", f"Authorization: Bearer {AGENT_MEMORY_API_KEY}",
        f"{AGENT_MEMORY_ENDPOINT.rstrip('/')}/v1/stores/{AGENT_MEMORY_STORE_ID}/session-memory/{SESSION_ID}",
    ],
    check=True,
    capture_output=True,
    text=True,
)

session = json.loads(retrieve.stdout)
events = session["events"]
print(f"Session: {session['sessionId']}")
print(f"Owner: {session['ownerId']}")
print(f"{len(events)} events\n")

for index, event in enumerate(events, start=1):
    text = "\n".join(part["text"] for part in event["content"])
    print(f"{index}. {event['role']} ({event['actorId']})")
    print(text)
    print()


Session: banking-session-5a9db27a
Owner: customer-5a9db27a
3 events

1. USER (customer-5a9db27a)
I'm trying to pull together a deposit for a place of my own. I'd like to be ready in about a year and a half, but freelance work makes that messy. Some months are great, and some are really quiet.

2. ASSISTANT (relationship-manager-agent)
That makes sense—freelance income can make an 18-month goal harder to plan around. For your next review, we can talk through the deposit amount you’re aiming for and how your income varies; do you have a rough target in mind?

3. USER (customer-5a9db27a)
I found a flat I like, but I want to keep the deposit goal flexible until my next busy freelance month.



## Step 3: Creating long-term memory

### Automatic extraction from session memory

Relevant details from session memory are automatically extracted into long-term memory by Redis Agent Memory. Long-term memory is durable, scoped by owner, and reusable across sessions.

Extraction runs on the extraction cadence, which defaults to 5 minutes. This setting can be configured in the Redis Cloud console. If the next cell shows no extracted memories, wait and run it again.

In [9]:
#NBVAL_SKIP
# Extraction runs in the background on the extraction cadence configured for the service.
# The application does not call extraction.
with AgentMemory(
    AGENT_MEMORY_ENDPOINT,
    store_id=AGENT_MEMORY_STORE_ID,
    api_key=AGENT_MEMORY_API_KEY,
) as memory_client:
    extracted = memory_client.search_long_term_memory(
        request=models.SearchLongTermMemoryRequestContent(
            text="What is this client's savings goal, timeline, and income constraint?",
            filter_=models.LongTermMemoryFilter(
                owner_id=models.OwnerIDFilter(eq=CUSTOMER_ID),
            ),
            limit=5,
        ),
    )

print(f"{len(extracted.items)} extracted memories for {CUSTOMER_ID}\n")
if not extracted.items:
    print("None yet. Extraction follows the extraction cadence, so run this cell again after that interval.")

for index, memory in enumerate(extracted.items, start=1):
    print(f"{index}. {memory.memory_type or 'memory'} ({memory.id})")
    print(memory.text)
    print()


5 extracted memories for customer-5a9db27a

1. semantic (f3c6b26076b342e0bd3f4128fd1a8381)
User wants to keep the deposit target flexible until the next busy freelance month.

2. episodic (9aebbb610c794264b7fb0a0602f35dd5)
The next financial review is intended to cover User's target deposit amount and the variability of User's freelance income.

3. episodic (3788c56c488c4e83ab2a7a028cc05a8d)
User is aiming to be ready to put down a deposit on a place of User's own in about 18 months, around April 2028.

4. semantic (2402825813744797956f116ea2025416)
User's freelance income varies significantly, with some months being very good and some months being very quiet.

5. episodic (4d1b6baf652e44a78aa480e18cadb0fa)
User found a flat that User likes.



### Creating long-term memories directly

Long-term memories can also be created directly, without automatic extraction from session memory. This is useful when a fact should be available before a session starts.

#### Via SDK

In [10]:
#NBVAL_SKIP
# A fact the application already knows. It was not said in this session,
# so extraction from the conversation would not produce it.
MEETING_PREFERENCE_MEMORY_ID = f"meeting-preference-{uuid4().hex}"

with AgentMemory(
    AGENT_MEMORY_ENDPOINT,
    store_id=AGENT_MEMORY_STORE_ID,
    api_key=AGENT_MEMORY_API_KEY,
) as memory_client:
    created = memory_client.bulk_create_long_term_memories(
        memories=[
            models.CreateMemoryRecord(
                id=MEETING_PREFERENCE_MEMORY_ID,
                text=(
                    "The client prefers morning meetings and wants a short written "
                    "summary afterward."
                ),
                owner_id=CUSTOMER_ID,
                memory_type="semantic",
                topics=["meetings"],
            ),
        ],
    )

print(f"Created long-term memories: {', '.join(created.created)}")


Created long-term memories: meeting-preference-53a31f75653f442384c7024cd450cf28


#### Via HTTP POST

In [11]:
#NBVAL_SKIP
REVIEW_CADENCE_MEMORY_ID = f"review-cadence-{uuid4().hex}"

payload = {
    "memories": [
        {
            "id": REVIEW_CADENCE_MEMORY_ID,
            "text": (
                "The client has worked with this relationship team since 2019 "
                "and usually reviews goals once a year."
            ),
            "memoryType": "semantic",
            "ownerId": CUSTOMER_ID,
            "topics": ["relationship"],
        }
    ],
}

add_memory = subprocess.run(
    [
        "curl",
        "--fail-with-body",
        "--silent",
        "--show-error",
        "--request", "POST",
        "--header", f"Authorization: Bearer {AGENT_MEMORY_API_KEY}",
        "--header", "Content-Type: application/json",
        "--data", json.dumps(payload),
        f"{AGENT_MEMORY_ENDPOINT.rstrip('/')}/v1/stores/{AGENT_MEMORY_STORE_ID}/long-term-memory",
    ],
    check=True,
    capture_output=True,
    text=True,
)

created = json.loads(add_memory.stdout)
print(f"Created long-term memories: {', '.join(created['created'])}")


Created long-term memories: review-cadence-e6c19f063bdf4af79f8427d1ffe4c98b


### Retrieving long-term memory

To retrieve long-term memory from Redis Agent Memory, query the server via SDK or HTTP POST requests. The response will include memories relevant to the query.

A long-term memory search can combine query text with filters. The Python SDK uses the same parameters in snake case, such as `similarity_threshold`, `filter_`, and `filter_op`.

| Parameter | What it does |
| --- | --- |
| `text` | Text used for semantic search. |
| `similarityThreshold` | Minimum similarity for a vector match, from 0 to 1. This is normalized cosine similarity. |
| `filter` | Limits which memories can match. |
| `filterOp` | How filter conditions combine. `all` requires every condition. `any` requires one. |
| `limit` | Maximum number of results. The default is 10 and the maximum is 100. |
| `pageToken` | Opaque token from a previous response, used to fetch the next page. |

A filter can use these fields:

| Field | Operators |
| --- | --- |
| `sessionId`, `ownerId`, `namespace`, `topics`, `memoryType` | `eq`, `ne`, `in`, `all` |
| `createdAt` | `gt`, `gte`, `lt`, `lte`, `eq` |
| `attributes` | Structured fields on a custom memory type. |

#### Via SDK

In [12]:
#NBVAL_SKIP
# Recall is scoped to this customer, so another client's memories are not returned.
with AgentMemory(
    AGENT_MEMORY_ENDPOINT,
    store_id=AGENT_MEMORY_STORE_ID,
    api_key=AGENT_MEMORY_API_KEY,
) as memory_client:
    recalled = memory_client.search_long_term_memory(
        request=models.SearchLongTermMemoryRequestContent(
            text="What should the next meeting know about this client?",
            filter_=models.LongTermMemoryFilter(
                owner_id=models.OwnerIDFilter(eq=CUSTOMER_ID),
            ),
            limit=10,
        ),
    )

print(f"{len(recalled.items)} long-term memories for {CUSTOMER_ID}\n")

for index, memory in enumerate(recalled.items, start=1):
    print(f"{index}. {memory.memory_type or 'memory'} ({memory.id})")
    print(memory.text)
    print()


7 long-term memories for customer-5a9db27a

1. semantic (meeting-preference-53a31f75653f442384c7024cd450cf28)
The client prefers morning meetings and wants a short written summary afterward.

2. semantic (review-cadence-e6c19f063bdf4af79f8427d1ffe4c98b)
The client has worked with this relationship team since 2019 and usually reviews goals once a year.

3. episodic (9aebbb610c794264b7fb0a0602f35dd5)
The next financial review is intended to cover User's target deposit amount and the variability of User's freelance income.

4. semantic (f3c6b26076b342e0bd3f4128fd1a8381)
User wants to keep the deposit target flexible until the next busy freelance month.

5. episodic (3788c56c488c4e83ab2a7a028cc05a8d)
User is aiming to be ready to put down a deposit on a place of User's own in about 18 months, around April 2028.

6. episodic (4d1b6baf652e44a78aa480e18cadb0fa)
User found a flat that User likes.

7. semantic (2402825813744797956f116ea2025416)
User's freelance income varies significantly, with

#### Via HTTP POST

In [13]:
#NBVAL_SKIP
payload = {
    "text": "What should the next meeting know about this client?",
    "filter": {
        "ownerId": {
            "eq": CUSTOMER_ID,
        },
    },
    "limit": 10,
}

retrieve = subprocess.run(
    [
        "curl",
        "--fail-with-body",
        "--silent",
        "--show-error",
        "--request", "POST",
        "--header", f"Authorization: Bearer {AGENT_MEMORY_API_KEY}",
        "--header", "Content-Type: application/json",
        "--data", json.dumps(payload),
        f"{AGENT_MEMORY_ENDPOINT.rstrip('/')}/v1/stores/{AGENT_MEMORY_STORE_ID}/long-term-memory/search",
    ],
    check=True,
    capture_output=True,
    text=True,
)

recalled = json.loads(retrieve.stdout)
memories = recalled["items"]
print(f"{len(memories)} long-term memories for {CUSTOMER_ID}\n")

for index, memory in enumerate(memories, start=1):
    print(f"{index}. {memory.get('memoryType', 'memory')} ({memory['id']})")
    print(memory["text"])
    print()


7 long-term memories for customer-5a9db27a

1. semantic (meeting-preference-53a31f75653f442384c7024cd450cf28)
The client prefers morning meetings and wants a short written summary afterward.

2. semantic (review-cadence-e6c19f063bdf4af79f8427d1ffe4c98b)
The client has worked with this relationship team since 2019 and usually reviews goals once a year.

3. episodic (9aebbb610c794264b7fb0a0602f35dd5)
The next financial review is intended to cover User's target deposit amount and the variability of User's freelance income.

4. semantic (f3c6b26076b342e0bd3f4128fd1a8381)
User wants to keep the deposit target flexible until the next busy freelance month.

5. episodic (3788c56c488c4e83ab2a7a028cc05a8d)
User is aiming to be ready to put down a deposit on a place of User's own in about 18 months, around April 2028.

6. episodic (4d1b6baf652e44a78aa480e18cadb0fa)
User found a flat that User likes.

7. semantic (2402825813744797956f116ea2025416)
User's freelance income varies significantly, with

## Step 4: Using long-term memory in a conversation

A later meeting uses a new session and the same customer. A small `CustomerSession` helper holds that pairing for the whole meeting. Type each client line at the prompt. The cell stores that turn, searches this owner's `semantic` long-term memories, and adds the returned text to the model context. After the agent replies, it stores the reply on the same session. Type `quit` to end the meeting.

Type a client message at the prompt, such as "Please schedule our next goals review and send the follow-up after." and "What do I have to prepare for the review?".

In [ ]:
#NBVAL_SKIP
from openai import OpenAI

FOLLOW_UP_SESSION_ID = f"follow-up-session-{RUN_ID}"

system_prompt_template = """You are a relationship-manager agent for a retail bank. You speak with clients about their goals and help them prepare for the next review.

Follow these rules:
- Respond in two or three sentences, in plain language.
- Use what the client says in this conversation and the long-term memories below.
- Do not invent account balances, transactions, product eligibility, or approval decisions. Those remain in the bank's systems of record.
- Do not give personalised financial, tax, or legal advice. If the client asks for a decision, say a banker needs to confirm it.

Known context for this client:
{memory_context}
"""


class CustomerSession:
    """Hold one customer and one session, so recall and record share the same scope."""

    def __init__(self, memory_client, owner_id, session_id, assistant_id):
        self.memory = memory_client
        self.owner_id = owner_id
        self.session_id = session_id
        self.assistant_id = assistant_id

    def recall(self, query_text, limit=10, memory_type="semantic"):
        recalled = self.memory.search_long_term_memory(
            request=models.SearchLongTermMemoryRequestContent(
                text=query_text,
                filter_=models.LongTermMemoryFilter(
                    owner_id=models.OwnerIDFilter(eq=self.owner_id),
                    memory_type=models.MemoryTypeFilter(eq=memory_type),
                ),
                limit=limit,
            ),
        )
        lines = [f"- {memory.text}" for memory in recalled.items]
        return "\n".join(lines) if lines else "- No long-term memories were found for this client."

    def record_user_turn(self, client_text):
        event = self.memory.add_session_event(
            session_id=self.session_id,
            actor_id=self.owner_id,
            role=models.MessageRole.USER,
            content=[models.Text(text=client_text)],
            created_at=datetime.now(timezone.utc),
        )

    def record_assistant_reply(self, reply):
        event = self.memory.add_session_event(
            session_id=self.session_id,
            actor_id=self.assistant_id,
            role=models.MessageRole.ASSISTANT,
            content=[models.Text(text=reply)],
            created_at=datetime.now(timezone.utc),
        )


openai_client = OpenAI(api_key=OPENAI_API_KEY)
history = [{"role": "system", "content": ""}]

with AgentMemory(
    AGENT_MEMORY_ENDPOINT,
    store_id=AGENT_MEMORY_STORE_ID,
    api_key=AGENT_MEMORY_API_KEY,
) as memory_client:
    customer_session = CustomerSession(memory_client, CUSTOMER_ID, FOLLOW_UP_SESSION_ID, ASSISTANT_ID)
    while True:
        client_text = input("What would you like to cover in this goals review? (or 'quit' to end): ").strip()
        if client_text.lower() in {"quit", "exit"} or not client_text:
            break

        # Store the client turn before recall, so the session holds it even if the model call fails.
        customer_session.record_user_turn(client_text)
        # Durable preferences only.
        memory_context = customer_session.recall(client_text, memory_type="semantic")
        history[0]["content"] = system_prompt_template.format(memory_context=memory_context)
        history.append({"role": "user", "content": client_text})

        reply = openai_client.chat.completions.create(
            model="gpt-6-luna",
            messages=history,
        ).choices[0].message.content

        history.append({"role": "assistant", "content": reply})
        print(f"Client: {client_text}")
        print(f"Agent: {reply}\n")
        customer_session.record_assistant_reply(reply)


Client: Please schedule our next goals review and send the follow-up after
Agent: Of course—what morning and date would work for your next goals review? I’ll arrange the meeting and send you a short written summary afterward.

Client: What do I have to prepare for the review?
Agent: Please think about any changes to your goals, your freelance income outlook—including your next busy month—and any questions you’d like to cover. We can keep the deposit target flexible until then; there’s no need to settle on a specific amount in advance.



Based on the agent output, this meeting used long-term memories for the same customer. Before each reply, the cell searched long-term memory by owner and placed the returned text in the system prompt. The client did not restate those facts here.

The replies draw on memories extracted from the earlier session and on preferences stored directly for this customer. The agent can continue the relationship from that context.

>NOTE: This cell keeps a local `history` list for the model messages for simplicity. Calling `get_session_memory` on every turn would be an extra network call, and this meeting stays inside one cell, so that call would only return turns the list already has.

A way to use the session memory is to call `get_session_memory` before the next agent turn and pass those events to the agent, so the application does not keep a separate conversation store. Use that call when this process does not already hold the turns, and the network call is the only way to get them:

- The next turn arrives as a new request, and the local list is gone.
- Another worker or service continues the same `sessionId`.
- The process restarted, and the stored events are all that remain.
- The session has been summarized. The session read returns that summary plus the recent events, while a local list still holds every original line.

## Cleanup: Deleting memories from Redis Agent Memory

#### Delete session memory

#### Via SDK

In [15]:
#NBVAL_SKIP
with AgentMemory(
    AGENT_MEMORY_ENDPOINT,
    store_id=AGENT_MEMORY_STORE_ID,
    api_key=AGENT_MEMORY_API_KEY,
) as memory_client:
    memory_client.delete_session_memory(session_id=SESSION_ID)

print(f"Deleted session memories from {SESSION_ID}")


Deleted session memories from banking-session-5a9db27a


#### Via HTTP DELETE


The follow-up session is deleted with `DELETE /v1/stores/{storeId}/session-memory/{sessionId}`. The response has no body.

In [16]:
#NBVAL_SKIP
delete_follow_up = subprocess.run(
    [
        "curl",
        "--fail-with-body",
        "--silent",
        "--show-error",
        "--request", "DELETE",
        "--header", f"Authorization: Bearer {AGENT_MEMORY_API_KEY}",
        f"{AGENT_MEMORY_ENDPOINT.rstrip('/')}/v1/stores/{AGENT_MEMORY_STORE_ID}/session-memory/{FOLLOW_UP_SESSION_ID}",
    ],
    check=True,
)

print(f"Deleted session memories from {FOLLOW_UP_SESSION_ID}")


Deleted session memories from follow-up-session-5a9db27a


#### Delete long-term memory

In [17]:
#NBVAL_SKIP
# The API deletes long-term memories by id. Collect this customer's records, then delete them.
memory_ids = []
page_token = None

with AgentMemory(
    AGENT_MEMORY_ENDPOINT,
    store_id=AGENT_MEMORY_STORE_ID,
    api_key=AGENT_MEMORY_API_KEY,
) as memory_client:
    while True:
        page = memory_client.search_long_term_memory(
            request=models.SearchLongTermMemoryRequestContent(
                filter_=models.LongTermMemoryFilter(
                    owner_id=models.OwnerIDFilter(eq=CUSTOMER_ID),
                ),
                limit=100,
                page_token=page_token,
            ),
        )
        memory_ids.extend(memory.id for memory in page.items)
        page_token = page.next_page_token
        if not page_token:
            break

    for start in range(0, len(memory_ids), 100):
        memory_client.bulk_delete_long_term_memories(
            memory_ids=memory_ids[start:start + 100],
        )

print(f"Deleted {len(memory_ids)} long-term memories for {CUSTOMER_ID}")


Deleted 9 long-term memories for customer-5a9db27a


## Conclusion

Redis Agent Memory allows an agent to hold the current conversation in session memory and recall a client's goals and preferences from long-term memory in a later meeting. In this notebook, we have created and retrieved long-term memories from a conversation turn, and used them in a follow-up meeting, through the Python SDK, HTTP requests, and the Redis Cloud Console.